# Fluss: Batch Log Scanner Demonstration

This notebook demonstrates how to use the `create_record_batch_log_scanner` functionality for high-performance, batch-based log scanning in Fluss. 

Key features covered:
- Setting up a table and writing data in batches.
- Using `create_record_batch_log_scanner()` for Arrow-native scanning.
- **Async Iteration**: Using `async for` to consume RecordBatches.
- **Streaming**: Running producer and consumer concurrently in a single notebook.

In [1]:
import fluss
import os
from datetime import datetime

# Get the exact path of the loaded compiled _fluss extension
extension_path = fluss._fluss.__file__
mod_time = os.path.getmtime(extension_path)

print(f"Loaded Extension: {extension_path}")
print(f"Compiled At: {datetime.fromtimestamp(mod_time)}")

Loaded Extension: /Users/jaredyu/Desktop/open_source/fluss-rust/bindings/python/fluss/_fluss.cpython-312-darwin.so
Compiled At: 2026-03-12 21:55:16.964055


In [2]:
import asyncio
import pyarrow as pa
# import fluss
import time

## 1. Setup Table

First, we connect to the Fluss cluster and create a test table with a simple schema.

In [45]:
async def setup_table():
    config = fluss.Config({"bootstrap.servers": "127.0.0.1:9123"})
    conn = await fluss.FlussConnection.create(config)
    admin = await conn.get_admin()
    
    table_path = fluss.TablePath("fluss", "batch_test_table")
    
    # Cleanup old table
    await admin.drop_table(table_path, ignore_if_not_exists=True)
    
    # Define schema
    schema = fluss.Schema(
        pa.schema([pa.field("id", pa.int32()), pa.field("value", pa.string())])
    )
    table_descriptor = fluss.TableDescriptor(schema, bucket_count=1)
    
    await admin.create_table(table_path, table_descriptor)
    print(f"Created table: {table_path}")
    return conn, table_path

# Note: In a Jupyter notebook, you can await directly in the cell
conn, table_path = await setup_table()

Created table: fluss.batch_test_table


## 2. Shared Producer Logic

We'll use this function to load data into the table. It appends records in batches of size 3.

In [46]:
async def producer(table, total_records=12, delay=0.5):
    writer = table.new_append().create_writer()
    pa_schema = pa.schema([pa.field("id", pa.int32()), pa.field("value", pa.string())])
    
    for i in range(0, total_records, 3):
        batch = pa.RecordBatch.from_arrays(
            [pa.array([i, i+1, i+2], type=pa.int32()), 
             pa.array([f"data_{i}", f"data_{i+1}", f"data_{i+2}"])],
            schema=pa_schema,
        )
        writer.write_arrow_batch(batch)
        await writer.flush()
        print(f"[Producer] Wrote batch starting at {i}")
        await asyncio.sleep(delay)

## 3. Batch Scanning with `async for`

This is the recommended way to consume data in a batch-oriented fashion. The `LogScanner` yielded by `create_record_batch_log_scanner()` is an async iterator that yields `RecordBatch` objects.

In [47]:
async def consumer_batch_iterator(table, expected_records=12):
    # Create a batch-oriented scanner
    scanner = await table.new_scan().create_record_batch_log_scanner()
    # Subscribe to bucket 0 from the beginning
    scanner.subscribe(bucket_id=0, start_offset=0)
    
    count = 0
    print("[Consumer] Starting async batch iteration...")
    
    async for rb in scanner:
        # 'rb' is a fluss.RecordBatch object containing metadata and the Arrow batch
        batch = rb.batch # This is the actual pyarrow.RecordBatch
        print(f"[Consumer] Received batch: rows={batch.num_rows}, offsets={rb.base_offset}-{rb.last_offset}")
        
        # You can process the batch natively with Arrow/Pandas
        df = batch.to_pandas()
        print(df)
        
        count += batch.num_rows
        if count >= expected_records:
            print(f"[Consumer] Reached expected count {count}. Stopping.")
            break

## 4. Single Notebook Streaming Demo

Combining the producer and consumer using `asyncio.gather`. This demonstrates that you don't need two notebooks; a single async runtime can handle both.

In [48]:
table = await conn.get_table(table_path)

# Run both concurrently
num_records = 60
await asyncio.gather(
    producer(table, total_records=num_records, delay=1),
    consumer_batch_iterator(table, expected_records=num_records)
)

[Consumer] Starting async batch iteration...
[Producer] Wrote batch starting at 0
[Consumer] Received batch: rows=3, offsets=0-2
   id   value
0   0  data_0
1   1  data_1
2   2  data_2
[Producer] Wrote batch starting at 3
[Consumer] Received batch: rows=3, offsets=3-5
   id   value
0   3  data_3
1   4  data_4
2   5  data_5
[Producer] Wrote batch starting at 6
[Consumer] Received batch: rows=3, offsets=6-8
   id   value
0   6  data_6
1   7  data_7
2   8  data_8
[Producer] Wrote batch starting at 9
[Consumer] Received batch: rows=3, offsets=9-11
   id    value
0   9   data_9
1  10  data_10
2  11  data_11
[Producer] Wrote batch starting at 12
[Consumer] Received batch: rows=3, offsets=12-14
   id    value
0  12  data_12
1  13  data_13
2  14  data_14
[Producer] Wrote batch starting at 15
[Consumer] Received batch: rows=3, offsets=15-17
   id    value
0  15  data_15
1  16  data_16
2  17  data_17
[Producer] Wrote batch starting at 18
[Consumer] Received batch: rows=3, offsets=18-20
   id    

[None, None]

## 5. Other Batch Methods

Besides async iteration, you can also poll for data explicitly or grab everything as a single Arrow table.

In [42]:
print("\n--- Demonstrating to_arrow() ---")
scanner = await table.new_scan().create_record_batch_log_scanner()
scanner.subscribe(bucket_id=0, start_offset=0)

# to_arrow() reads all available data into a single PyArrow Table
arrow_table = scanner.to_arrow()
print(f"Total rows in table: {arrow_table.num_rows}")
print(arrow_table.to_pandas().head())

print("\n--- Demonstrating poll_arrow() ---")
scanner2 = await table.new_scan().create_record_batch_log_scanner()
scanner2.subscribe(bucket_id=0, start_offset=0)

# poll_arrow() fetches a chunk of data (merged batches) or returns an empty table on timeout
poll_result = scanner2.poll_arrow(timeout_ms=1000)
print(f"Polled {poll_result.num_rows} rows")


--- Demonstrating to_arrow() ---
Total rows in table: 60
   id   value
0   0  data_0
1   1  data_1
2   2  data_2
3   3  data_3
4   4  data_4

--- Demonstrating poll_arrow() ---
Polled 60 rows


## 6. Cleanup

Always close the connection when done.

In [44]:
conn.close()
print("Connection closed.")

Connection closed.
